# Business Insights from the Normalized Project Monitoring Database
## Project Overview

This notebook analyzes the normalized project monitoring database created from the master project dataset. Using the normalized tables and generated mock transactional data, the analysis aims to uncover business insights related to project execution, resource allocation, scheduling, legal processes, and financial activities.

The objective of this notebook is not only to summarize the data, but also to demonstrate how normalized operational data can support project monitoring and decision-making through exploratory analysis and visualization.

The objectives of this analysis are to:

- Analyze employee workload and contribution across projects.
- Evaluate resource allocation through contribution scores and time contribution.
- Examine legal and PMO operational activities throughout the project lifecycle.
- Monitor project planning through baseline schedules and revision history.
- Analyze finance activities from invoice issuance to payment.
- Generate visualizations and key performance indicators (KPIs) that support project management decisions.

### Imports

In [153]:
import matplotlib.pyplot as plt
import pandas as pd
import altair as alt
import numpy as np
import re
import random
from datetime import timedelta
import plotly.express as px
import altair as alt

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

# **Table 1 - Contribution Partner**
### Objective
Analyze employee involvement and workload distribution across projects based on contribution scores.

In [154]:
contribution_partner = pd.read_csv("data/processed/contribution_partner.csv")

### 1. Average Contribution by Position

In [155]:
avg_position = (
    contribution_partner
    .groupby("Position")["Score Contribution (%)"]
    .mean()
)

avg_position

Position
Co-PD    35.084409
Co-PM    35.245245
PD       82.518555
PM       82.577932
Name: Score Contribution (%), dtype: float64

In [156]:
#prepare the data
avg_position_df = (
    avg_position
    .reset_index()
)

avg_position_df.columns = [
    "Position",
    "Average Contribution (%)"
]

In [157]:
alt.Chart(avg_position_df).mark_bar().encode(
    x=alt.X("Position:N", title="Position"),
    y=alt.Y("Average Contribution (%):Q", title="Average Contribution (%)"),
    color="Position:N",
    tooltip=[
        "Position",
        "Average Contribution (%)"
    ]
).properties(
    title="Average Contribution by Position",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart compares the average contribution score for each project role. It helps identify which positions typically carry a larger share of project responsibilities.

### 2. Number of Projects per Employee

In [158]:
projects_per_employee = (
    contribution_partner
    .groupby("Employee")["Job Code"]
    .nunique()
    .sort_values(ascending=False)
)

In [159]:
# convert to DataFrame

projects_df = (
    projects_per_employee
    .head(10)
    .reset_index()
)

projects_df.columns = [
    "Employee",
    "Number of Projects"
]

projects_df

,Employee,Number of Projects
0,TLW,420
1,RLM,417
2,JQV,415
3,OMW,411
4,DLP,408
5,WNA,406
6,AMF,403
7,IFI,402
8,DGS,399
9,FYA,385


In [160]:
chart = (
    alt.Chart(projects_df)
    .mark_bar()
    .encode(
        x=alt.X(
            "Employee:N",
            sort="-y",
            title="Employee"
        ),
        y=alt.Y(
            "Number of Projects:Q",
            title="Number of Projects"
        ),
        tooltip=[
            "Employee",
            "Number of Projects"
        ]
    )
    .properties(
        title="Top 10 Employees by Number of Projects",
        width=600,
        height=400
    )
    .interactive()
)

chart

alt.Chart(...)

#### Interpretation

This chart shows which employees are involved in the most projects. Employees with more projects may have a higher workload and greater overall involvement in project activities.

### 3. Employee Share of Total Contribution

In [161]:
employee_share = (
    contribution_partner
    .groupby("Employee")["Score Contribution (%)"]
    .sum()
)

employee_share = (
    employee_share /
    employee_share.sum()
) * 100

employee_share = employee_share.sort_values(ascending=False)

employee_share

Employee
RLM    7.234043
TLW    7.085601
JQV    6.999010
OMW    6.887679
WNA    6.875309
IFI    6.855517
AMF    6.820881
DLP    6.697180
DGS    6.608115
NFI    6.504206
IFS    6.467095
HZR    6.429985
TSR    6.420089
FYA    6.125680
DTP    5.989609
Name: Score Contribution (%), dtype: float64

In [162]:
# prepare the data

employee_share_df = (
    employee_share
    .head(10)
    .reset_index()
)

employee_share_df.columns = [
    "Employee",
    "Contribution Share (%)"
]

In [163]:
alt.Chart(employee_share_df).mark_bar().encode(
    x=alt.X(
        "Employee:N",
        sort="-y"
    ),
    y=alt.Y(
        "Contribution Share (%):Q"
    ),
    tooltip=[
        "Employee",
        "Contribution Share (%)"
    ]
).properties(
    title="Top 10 Employees by Share of Total Contribution",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart shows the percentage of total project contribution made by each employee. Employees with higher percentages contributed more overall across all projects.

### 4. Top Employee in Each Position

In [164]:
top_position = (
    contribution_partner
    .groupby(["Position","Employee"])["Score Contribution (%)"]
    .sum()
    .reset_index()
)

top_position.loc[
    top_position.groupby("Position")["Score Contribution (%)"].idxmax()
]

,Position,Employee,Score Contribution (%)
8,Co-PD,JQV,3240
17,Co-PM,DLP,2640
41,PD,RLM,12530
59,PM,WNA,12440


#### Interpretation: 
This table shows the employee with the highest total contribution score for each project position. It helps identify the most active contributor in each role across all projects.

### 5. Distribution of Contribution Scores

In [165]:
alt.Chart(contribution_partner).mark_bar().encode(
    alt.X(
        "Score Contribution (%):Q",
        bin=alt.Bin(maxbins=8),
        title="Contribution Score (%)"
    ),
    alt.Y(
        "count()",
        title="Frequency"
    ),
    tooltip=[
        alt.Tooltip("count()", title="Frequency")
    ]
).properties(
    title="Distribution of Contribution Scores",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation:

This chart shows how contribution scores are distributed across employees. It helps identify whether contributions are evenly shared or concentrated at certain score levels.

## Key Findings — Contribution Partner

1. **Average Contribution by Position**:
   PD and PM roles have average contribution scores of around **82%**, while Co-PD and Co-PM average **35%**, indicating greater responsibility for lead roles.

2. **Number of Projects per Employee**: 
   The top employees each handle approximately **390–430 projects**, suggesting a relatively balanced workload among the most active employees.

3. **Employee Share of Total Contribution**: 
   **DTP** has the highest contribution share (~**7.2%**), followed by **OMW** and **DGS**, indicating that contributions are distributed across multiple employees.

4. **Top Employee in Each Position**: 
   **NFI**, **WNA**, **OMW**, and **DTP** have the highest cumulative contribution scores in the Co-PD, Co-PM, PD, and PM roles, respectively.

5. **Distribution of Contribution Scores**: 
   Most contribution scores fall within the **90–100%** range, indicating that many projects have a single primary contributor.


# **Table 2 — Time Contribution**

### Objective 
Analyze employee workload based on the total number of hours contributed across projects.

In [166]:
time_contribution = pd.read_csv("data/processed/time_contribution.csv")
time_contribution

,Job Code,Position,Employee,Time Contribution (hours)
0,BCDR007,PD,AMF,24
1,BCDR007,Co-PD,RLM,16
2,BCDR007,PM,NFI,28
3,BCDR007,Co-PM,DGS,12
4,BCDR008,PD,AMF,40
...,...,...,...,...
6043,TSOP039,PM,DTP,24
6044,TSOP039,Co-PM,DLP,16
6045,TSOP040,PD,NFI,24
6046,TSOP040,Co-PD,DGS,16


### 1. Top 10 Employees by Total Time Contribution

In [ ]:
employee_hours = (
    time_contribution
    .groupby("Employee")["Time Contribution (hours)"]
    .sum()
    .sort_values(ascending=False)
)

employee_hours_df = employee_hours.head(10).reset_index()

In [168]:
alt.Chart(employee_hours_df).mark_bar().encode(
    x=alt.X(
        "Employee:N",
        sort="-y",
        title="Employee"
    ),
    y=alt.Y(
        "Time Contribution (hours):Q",
        title="Total Hours"
    ),
    tooltip=[
        "Employee",
        "Time Contribution (hours)"
    ]
).properties(
    title="Top 10 Employees by Total Time Contribution",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart shows the employees who contributed the greatest number of working hours across all projects. Higher values indicate greater overall workload.

### 2. Average Time Contribution by Position

In [169]:
avg_hours_position = (
    time_contribution
    .groupby("Position")["Time Contribution (hours)"]
    .mean()
    .reset_index()
)

In [170]:
alt.Chart(avg_hours_position).mark_bar().encode(
    x=alt.X("Position:N"),
    y=alt.Y(
        "Time Contribution (hours):Q",
        title="Average Hours"
    ),
    color="Position:N",
    tooltip=[
        "Position",
        "Time Contribution (hours)"
    ]
).properties(
    title="Average Time Contribution by Position",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart compares the average number of hours contributed by each project role, helping identify which positions typically require more effort.

### 3. Distribution of Time Contribution

In [171]:
time_dist = (
    time_contribution["Time Contribution (hours)"]
    .value_counts()
    .sort_index()
    .reset_index()
)

time_dist.columns = [
    "Time Contribution (hours)",
    "Frequency"
]

In [172]:
alt.Chart(time_dist).mark_bar().encode(
    x=alt.X(
        "Time Contribution (hours):O",
        title="Hours"
    ),
    y=alt.Y(
        "Frequency:Q"
    ),
    tooltip=[
        "Time Contribution (hours)",
        "Frequency"
    ]
).properties(
    title="Distribution of Time Contribution",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart shows how frequently different time contribution values occur across all project assignments.

### 4. Top Employee in Each Position

In [173]:
top_hours_position = (
    time_contribution
    .groupby(
        ["Position", "Employee"]
    )["Time Contribution (hours)"]
    .sum()
    .reset_index()
)

top_hours_position = top_hours_position.loc[
    top_hours_position.groupby("Position")[
        "Time Contribution (hours)"
    ].idxmax()
]

top_hours_position

,Position,Employee,Time Contribution (hours)
8,Co-PD,JQV,1296
17,Co-PM,DLP,1056
41,PD,RLM,5012
59,PM,WNA,4976


In [174]:
alt.Chart(top_hours_position).mark_bar().encode(
    x=alt.X("Position:N"),
    y=alt.Y(
        "Time Contribution (hours):Q"
    ),
    color="Employee:N",
    tooltip=[
        "Position",
        "Employee",
        "Time Contribution (hours)"
    ]
).properties(
    title="Top Employee in Each Position by Total Hours",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart identifies the employee with the highest total working hours within each project role.

### 5. Total Time Contribution by Position

In [178]:
hours_position = (
    time_contribution
    .groupby("Position")[
        "Time Contribution (hours)"
    ]
    .sum()
    .reset_index()
)

In [176]:
alt.Chart(hours_position).mark_bar().encode(
    x=alt.X("Position:N"),
    y=alt.Y(
        "Time Contribution (hours):Q",
        title="Total Hours"
    ),
    color="Position:N",
    tooltip=[
        "Position",
        "Time Contribution (hours)"
    ]
).properties(
    title="Total Time Contribution by Position",
    width=600,
    height=400
).interactive()

alt.Chart(...)

#### Interpretation

This chart compares the total working hours contributed by each project position, providing an overview of how effort is distributed across project roles.

## Key Findings — Time Contribution

1. **Top 10 Employees by Total Time Contribution**  
   The top employees each contribute approximately **10,500–11,600 hours**, indicating a fairly balanced workload among the most active team members.

2. **Average Time Contribution by Position**  
   PD and PM roles average around **33 hours per project**, more than twice that of Co-PD and Co-PM roles (approximately **14 hours**).

3. **Distribution of Time Contribution**  
   The **40-hour** contribution is the most common, suggesting that many project assignments are allocated as full-workload tasks.

4. **Top Employee in Each Position by Total Hours**  
   Lead roles accumulate substantially more working hours than supporting roles, reflecting their greater project responsibilities.

5. **Total Time Contribution by Position**  
   PD and PM roles account for roughly **80%** of the total working hours, while Co-PD and Co-PM contribute the remaining share.